In [10]:
import pandas as pd
import scipy.sparse as ss
import numpy as np
from sklearn.decomposition import TruncatedSVD
import sklearn.manifold
#import tsne  # or whatever tsne package you chose
from sklearn.manifold import TSNE  # Use sklearn's implementation
import re

In [4]:
raw_data = pd.read_csv('subreddit-overlap')
raw_data.head()

In [7]:
subreddit_popularity = raw_data.groupby('t2_subreddit')['NumOverlaps'].sum()
subreddits = np.array(subreddit_popularity.sort_values(ascending=False).index)
index_map = dict(np.vstack([subreddits, np.arange(subreddits.shape[0])]).T)

count_matrix = ss.coo_matrix((raw_data.NumOverlaps, 
                              (raw_data.t2_subreddit.map(index_map),
                               raw_data.t1_subreddit.map(index_map))),
                             shape=(subreddits.shape[0], subreddits.shape[0]),
                             dtype=np.float64)
count_matrix


<COOrdinate sparse matrix of dtype 'float64'
	with 15381950 stored elements and shape (56187, 56187)>

In [11]:
conditional_prob_matrix = count_matrix.tocsr()
row_sums = np.array(conditional_prob_matrix.sum(axis=1))[:,0]
row_indices, col_indices = conditional_prob_matrix.nonzero()
conditional_prob_matrix.data /= row_sums[row_indices]

reduced_vectors = TruncatedSVD(n_components=500,
                               random_state=0).fit_transform(conditional_prob_matrix)
reduced_vectors /= np.sqrt((reduced_vectors**2).sum(axis=1))[:, np.newaxis]
# Replace the tsne.bh_sne call with sklearn's TSNE
tsne_model = TSNE(n_components=2, 
                  perplexity=50.0, 
                  random_state=0,
                  method='barnes_hut',  # equivalent to bh_sne
                  n_iter=1000)
subreddit_map = tsne_model.fit_transform(reduced_vectors[:10000])

subreddit_map_df = pd.DataFrame(subreddit_map, columns=('x', 'y'))
subreddit_map_df['subreddit'] = subreddits[:10000]
subreddit_map_df.head()

/opt/homebrew/Caskroom/miniconda/base/envs/cluster/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


,x,y,subreddit
0,12.893202,15.509626,AskReddit
1,12.998003,15.149703,pics
2,12.757484,15.289566,funny
3,13.624094,15.036378,todayilearned
4,15.621511,14.486432,worldnews


In [14]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_samples=5, 
                            min_cluster_size=20).fit(subreddit_map)
cluster_ids = clusterer.labels_
subreddit_map_df['cluster'] = cluster_ids

/opt/homebrew/Caskroom/miniconda/base/envs/cluster/lib/python3.13/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/cluster/lib/python3.13/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [18]:
for cid in range(cluster_ids.max() + 1):
    subreddits = subreddit_map_df.subreddit[cluster_ids == cid]
    subreddits = subreddits.values
        
    print('\nCluster {}:\n{}\n'.format(cid, subreddits))



Cluster 0:
['gaybros' 'askgaybros' 'rupaulsdragrace' 'gaymers' 'gay' 'ladybonersgw'
 'MassiveCock' 'GaybrosGoneWild' 'gaybrosgonemild' 'TotallyStraight'
 'penis' 'twinks' 'broslikeus' 'lolgrindr' 'TopsAndBottoms'
 'gaymersgonewild' 'gaybears' 'gaystoriesgonewild' 'mangonewild'
 'gaypornhunters' 'bigonewild' 'cock' 'tinydick' 'AsianLadyboners'
 'GayKink' 'RedditorCum' 'jacking' 'ratemycock' 'GaymersGoneMild'
 'lovegaymale' 'Balls' 'Bulges' 'FMN' 'softies' 'BeardPorn'
 'chesthairporn' 'DadsGoneWild' 'foreskin' 'forearmporn' 'selfservice'
 'PublicBoys' 'manass' 'gaynsfw' 'GayGifs' 'gaybroscirclejerk'
 'MaleArmpits' 'MalesMasturbating' 'ThickDick' 'gaycumsluts'
 'LGBTeensGoneMild' 'IWantToSuckCock' 'autofellatio' 'NSFW_GAY'
 'boysgonewild' 'MenWithToys' 'Blackdick' 'gayyoungold' 'manlove'
 'NSFW_DICK_and_Cock' 'GuysFromBehind' 'hotguyswithtattoos' 'bibros'
 'CuteGuyButts' 'gaysian']


Cluster 1:
['NASCAR' 'simracing' 'motogp' 'INDYCAR' 'dirtgame' 'wec' 'iRacing' 'USCR'
 'assettocorsa' 'pc

In [22]:
import pandas as pd
import numpy as np

# Read the original CSV file
df = pd.read_csv('subreddit-attrib.csv')

# Add cluster_id column with default empty values (object dtype to handle mixed types)
df['cluster_id'] = ''

# Create a mapping dictionary from subreddit name to cluster_id
cluster_mapping = {}

# Build the mapping from your clustering results
for cid in range(cluster_ids.max() + 1):
    subreddits = subreddit_map_df.subreddit[cluster_ids == cid]
    subreddits = subreddits.values
    
    # Add each subreddit to the mapping (store as integer)
    for subreddit in subreddits:
        cluster_mapping[subreddit] = int(cid)  # Convert to int
    
    print(f'Cluster {cid}: {len(subreddits)} subreddits')

# Update the cluster_id column based on the name column
# Only update rows where there's a mapping, leave others as empty string
df.loc[df['name'].isin(cluster_mapping.keys()), 'cluster_id'] = df.loc[df['name'].isin(cluster_mapping.keys()), 'name'].map(cluster_mapping)

# Check how many got assigned
assigned_count = (df['cluster_id'] != '').sum()
total_count = len(df)
print(f"\nAssigned clusters to {assigned_count} out of {total_count} subreddits")
print(f"Unassigned: {total_count - assigned_count}")

# Save to new CSV file
df.to_csv('../kosmos-recommend/subreddit-attrib-with-clusters.csv', index=False)
print("\nSaved to: subreddit-attrib-with-clusters.csv")

# Show sample of results
print("\nSample of updated data:")
print(df[['name', 'cluster_id']].head(10))

# Show some cluster assignments
print("\nSample cluster assignments:")
for cid in range(min(3, cluster_ids.max() + 1)):
    cluster_subreddits = df[df['cluster_id'] == cid]['name'].values
    print(f"Cluster {cid}: {cluster_subreddits[:5]}...")  # Show first 5

Cluster 0: 64 subreddits
Cluster 1: 23 subreddits
Cluster 2: 24 subreddits
Cluster 3: 26 subreddits
Cluster 4: 69 subreddits
Cluster 5: 21 subreddits
Cluster 6: 25 subreddits
Cluster 7: 39 subreddits
Cluster 8: 21 subreddits
Cluster 9: 25 subreddits
Cluster 10: 28 subreddits
Cluster 11: 23 subreddits
Cluster 12: 43 subreddits
Cluster 13: 35 subreddits
Cluster 14: 79 subreddits
Cluster 15: 141 subreddits
Cluster 16: 42 subreddits
Cluster 17: 50 subreddits
Cluster 18: 55 subreddits
Cluster 19: 30 subreddits
Cluster 20: 64 subreddits
Cluster 21: 240 subreddits
Cluster 22: 26 subreddits
Cluster 23: 36 subreddits
Cluster 24: 24 subreddits
Cluster 25: 59 subreddits
Cluster 26: 58 subreddits
Cluster 27: 33 subreddits
Cluster 28: 66 subreddits
Cluster 29: 21 subreddits
Cluster 30: 31 subreddits
Cluster 31: 45 subreddits
Cluster 32: 174 subreddits
Cluster 33: 48 subreddits
Cluster 34: 26 subreddits
Cluster 35: 25 subreddits
Cluster 36: 39 subreddits
Cluster 37: 82 subreddits
Cluster 38: 21 subr